# Experiments

### Setup

In [ ]:
# You can set them inline
# import os
# os.environ["OPENAI_API_KEY"] = ""
# os.environ["LANGSMITH_API_KEY"] = ""
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [14]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

这是我们在整个课程中一直使用的 RAG 应用程序。

In [20]:
import os
import tempfile
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.sitemap import SitemapLoader
from langchain_community.vectorstores import SKLearnVectorStore
from langchain_community.embeddings import DashScopeEmbeddings
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio

# TODO: Configure this model!
MODEL_NAME = "qwen3-max"
MODEL_PROVIDER = "qwen"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

def get_vector_db_retriever():
    persist_path = os.path.join(tempfile.gettempdir(), "union.parquet")
    embd = DashScopeEmbeddings(model="text-embedding-v4")

    # If vector store exists, then load it
    if os.path.exists(persist_path):
        vectorstore = SKLearnVectorStore(
            embedding=embd,
            persist_path=persist_path,
            serializer="parquet"
        )
        return vectorstore.as_retriever(lambda_mult=0)

    # Otherwise, index LangSmith documents and create new vector store
    ls_docs_sitemap_loader = SitemapLoader(web_path="https://docs.smith.langchain.com/sitemap.xml", continue_on_failure=True)
    ls_docs = ls_docs_sitemap_loader.load()

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=500, chunk_overlap=0
    )
    doc_splits = text_splitter.split_documents(ls_docs)

    vectorstore = SKLearnVectorStore.from_documents(
        documents=doc_splits,
        embedding=embd,
        persist_path=persist_path,
        serializer="parquet"
    )
    vectorstore.persist()
    return vectorstore.as_retriever(lambda_mult=0)

nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- Returns documents fetched from a vectorstore based on the user's question
"""
@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

"""
generate_response
- Calls `call_openai` to generate a model response after formatting inputs
"""
@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

"""
call_openai
- Returns the chat completion output from OpenAI
"""
@traceable(
    run_type="llm",
    metadata={
        "ls_provider": MODEL_PROVIDER,
        "ls_model_name": MODEL_NAME
    }
)
def call_openai(messages: List[dict]) -> str:
    return openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

"""
langsmith_rag
- Calls `retrieve_documents` to fetch documents
- Calls `generate_response` to generate a response based on the fetched documents
- Returns the model response
"""
@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


### Experiment

这里有几个重要组成部分:

1. 我们已经定义了一个评估器
2. 我们使用目标函数，将数据集示例（字典）传递给函数 `langsmith_rag` 所接受的输入类型（字符串）。

In [16]:
from langsmith import evaluate, Client

client = Client()
dataset_name = "RAG Application Golden Dataset"

def is_concise_enough(reference_outputs: dict, outputs: dict) -> dict:
    score = len(outputs["output"]) < 1.5 * len(reference_outputs["output"])
    return {"key": "is_concise", "score": int(score)}

def target_function(inputs: dict):
    return langsmith_rag(inputs["question"])

In [17]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="qwen3-max"
)

View the evaluation results for experiment: 'qwen3-max-0cb4af19' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=e09d451e-74cd-4724-9c8f-1ad0ab501815




12it [00:44,  3.71s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,"To set up tracing to LangSmith with LangChain,...",1,3.612492,005e9c46-2975-43b6-b751-f64e69357e7b,019b11d4-1143-7236-95ef-e878269e3723
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation. It...",None,"Yes, LangSmith supports offline evaluation thr...",1,3.468964,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11d4-1f63-73f3-b112-3d231df505d4
2,How can I trace with the @traceable decorator?,"To trace with the `@traceable` decorator, simp...",None,To trace with the @traceable decorator in Pyth...,1,4.083163,461abe28-c577-4c12-8577-85f68f4c44ab,019b11d4-2cf0-73d8-b3cb-5694a0520bff
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents....",None,"Yes, LangSmith can be used to evaluate agents....",1,3.546512,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11d4-3cf3-7084-8316-929804459f21
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,4.357607,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11d4-4ad1-710b-8dba-59dc097a6aa7
5,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,5.956372,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11d4-5bd9-75ec-ac88-c2914760122c
6,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript LangSmith SDK. It i...",None,"Yes, thers is a JavaScript LangSmith SDK!",0,3.136401,6fd631fb-c1d0-4030-9511-5bcd93b980b8,019b11d4-731e-73e7-9a3e-a9ac5815bbfb
7,How do I create user feedback with the LangSmi...,To create user feedback with the LangSmith SDK...,None,To create user feedback with the LangSmith SDK...,1,4.192185,0e14ba18-c937-4a43-b88b-678a1c5ea857,019b11d4-7f62-750d-ac8d-74aeef95d6a0
8,What testing capabilities does LangSmith have?,LangSmith offers prompt testing with built-in ...,None,LangSmith offers capabilities for creating dat...,1,2.426702,36415f3d-97ba-40a4-a75a-ecc563de722e,019b11d4-8fc9-7536-adf7-48058518986b
9,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,To set up tracing to LangSmith while using Lan...,1,3.578060,480d8cbc-c3d8-48d6-84ea-1366a59895be,019b11d4-9953-7530-9025-0ffae1742db5


### Modifying your Application

现在，让我们把模型换成 `qwen-plus`，看看它的性能如何！

进行此项更改，然后运行此代码片段！

In [21]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="qwen-plus"
)

View the evaluation results for experiment: 'qwen-plus-54cf6bef' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=3d3a122f-6f3e-4d41-934c-c9e05f2ea9b2




12it [00:35,  2.96s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,"To set up tracing to LangSmith with LangChain,...",1,2.989097,005e9c46-2975-43b6-b751-f64e69357e7b,019b11d8-6b1e-7317-ae4b-f6a05d69ae4f
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation for...",None,"Yes, LangSmith supports offline evaluation thr...",1,2.380970,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11d8-76d9-7167-8c16-8fdb15f3de2b
2,How can I trace with the @traceable decorator?,"To trace with the `@traceable` decorator, simp...",None,To trace with the @traceable decorator in Pyth...,1,2.584965,461abe28-c577-4c12-8577-85f68f4c44ab,019b11d8-8032-70c5-b060-67a45321a850
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",None,"Yes, LangSmith can be used to evaluate agents....",0,2.900454,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11d8-8a4b-74dc-bb1e-1e706fea5590
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,2.557435,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11d8-95a0-74f0-a1ff-eaa79f06717b
5,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,3.014704,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11d8-9f9d-750d-875a-49c463f84275
6,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript LangSmith SDK. It a...",None,"Yes, thers is a JavaScript LangSmith SDK!",0,2.639763,6fd631fb-c1d0-4030-9511-5bcd93b980b8,019b11d8-ab64-757b-8f37-289e17d21b78
7,How do I create user feedback with the LangSmi...,You can create user feedback using the `create...,None,To create user feedback with the LangSmith SDK...,1,3.633685,0e14ba18-c937-4a43-b88b-678a1c5ea857,019b11d8-b5b5-73b5-89ab-5f6cd99d11ea
8,What testing capabilities does LangSmith have?,LangSmith offers prompt testing with built-in ...,None,LangSmith offers capabilities for creating dat...,1,3.181939,36415f3d-97ba-40a4-a75a-ecc563de722e,019b11d8-c3e8-757d-9622-10dac36de9bf
9,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,To set up tracing to LangSmith while using Lan...,0,4.325355,480d8cbc-c3d8-48d6-84ea-1366a59895be,019b11d8-d056-751b-8582-b63654285b88


### 处理不同的数据片段

##### Dataset Version

可以使用 `list_examples` 函数中的 `as_of` 参数，在 SDK 中针对特定版本的数据集执行实验。

我们先尝试仅使用初始数据集运行实验。

In [22]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, as_of="initial dataset"),   # 使用 as_of 来指定版本
    evaluators=[is_concise_enough],
    experiment_prefix="initial dataset version"
)

View the evaluation results for experiment: 'initial dataset version-e07ef975' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=2c296cb7-e821-4138-aba9-609ed574a166




10it [00:29,  2.92s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation for...",None,"Yes, LangSmith supports offline evaluation thr...",1,2.213066,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11de-db0d-7363-9030-e38aa59d9c4a
1,How do I create user feedback with the LangSmi...,You can create user feedback using the `create...,None,To create user feedback with the LangSmith SDK...,1,3.220807,0e14ba18-c937-4a43-b88b-678a1c5ea857,019b11de-e3b9-70ce-9783-fb78fd558576
2,What testing capabilities does LangSmith have?,LangSmith offers prompt testing with built-in ...,None,LangSmith offers capabilities for creating dat...,1,3.192186,36415f3d-97ba-40a4-a75a-ecc563de722e,019b11de-f052-7759-a8d6-fc8d8ad79202
3,How can I trace with the @traceable decorator?,You can trace a function by decorating it with...,None,To trace with the @traceable decorator in Pyth...,1,2.622778,461abe28-c577-4c12-8577-85f68f4c44ab,019b11de-fcca-7685-a8fe-9f456a5f3698
4,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,To set up tracing to LangSmith while using Lan...,1,2.932631,480d8cbc-c3d8-48d6-84ea-1366a59895be,019b11df-071c-72b0-8297-c8e8acbeab57
5,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",None,"Yes, LangSmith can be used to evaluate agents....",0,3.104759,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11df-1297-7153-afe6-490d19e76c6b
6,How do I pass metadata in with @traceable?,You can pass metadata statically by including ...,None,You can pass metadata with the @traceable deco...,1,2.849445,9f7c3004-23ab-4f45-9c97-4a28a8c6b53a,019b11df-1ec7-734b-a3ae-2fd347bb3c9f
7,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,2.074259,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11df-29eb-7093-8451-a8b8e83d21a3
8,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,3.345999,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11df-3208-7203-8446-52f7c01d2c5e
9,Does LangSmith support online evaluation?,"Yes, LangSmith supports online evaluation to m...",None,"Yes, LangSmith supports online evaluation as a...",1,3.015194,f046f759-bda2-44a7-8683-9dd2fe186476,019b11df-3f1c-7285-94bc-38c9bb4a4811


##### Dataset Split

你可以对数据集的特定拆分部分进行实验，我们来尝试对“关键示例”拆分部分进行实验。

In [23]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, splits=["Crucial Examples"]),  # 传入一个拆分列表
    evaluators=[is_concise_enough],
    experiment_prefix="Crucial Examples split"
)

View the evaluation results for experiment: 'Crucial Examples split-319a718c' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=5ab6dc7e-4a82-4dc1-91d6-90aa18ddcf93




6it [00:16,  2.71s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,"To set up tracing to LangSmith with LangChain,...",1,3.183014,005e9c46-2975-43b6-b751-f64e69357e7b,019b11df-79fa-7231-b5f1-28367236ad52
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation for...",None,"Yes, LangSmith supports offline evaluation thr...",1,2.451789,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11df-8674-76f4-b12e-177674c9c8a5
2,How can I trace with the @traceable decorator?,"To trace with the `@traceable` decorator, simp...",None,To trace with the @traceable decorator in Pyth...,1,2.507686,461abe28-c577-4c12-8577-85f68f4c44ab,019b11df-900d-758d-8c01-6b044f622a50
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",None,"Yes, LangSmith can be used to evaluate agents....",1,2.636723,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11df-99d9-727f-810c-3489ae8aa638
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,2.032647,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11df-a425-72af-91f5-62bbce553ce8
5,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,2.877383,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11df-ac16-737e-ba4b-ebd0caf34039


##### Specific Data Points

你也可以指定要运行实验的单个数据点。

In [24]:
evaluate(
    target_function,
    data=client.list_examples(
        dataset_name=dataset_name, 
        example_ids=[   # 传入一个特定的 example_ids 列表
            # TODO: 你需要粘贴你自己的 example ids 才能使其正常工作！
            "e4d4a700-41e5-4dce-880c-c5fd17f26e75",
            "9f7c3004-23ab-4f45-9c97-4a28a8c6b53a"
        ]
    ),
    evaluators=[is_concise_enough],
    experiment_prefix="two specific example ids"
)

View the evaluation results for experiment: 'two specific example ids-39bd1314' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=43a91b73-2a33-4140-9967-0747418ce673




2it [00:06,  3.49s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,You can pass metadata statically by including ...,None,You can pass metadata with the @traceable deco...,1,2.129799,9f7c3004-23ab-4f45-9c97-4a28a8c6b53a,019b11df-ddf4-70ba-acd6-2c5e87f8d99d
1,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,4.268314,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11df-e649-77f8-b7f1-4ccbd6807491


### Other Parameters

##### Repetitions

你可以多次进行实验，以确保获得一致的结果。

In [25]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="two repetitions",
    num_repetitions=2   # This field defaults to 1
)

View the evaluation results for experiment: 'two repetitions-b686367a' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=26dc2542-2240-4116-b7da-aca6295e2913




24it [01:09,  2.89s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,"To set up tracing to LangSmith with LangChain,...",1,3.455725,005e9c46-2975-43b6-b751-f64e69357e7b,019b11e0-32b5-76e2-9c09-9e3fdefc7e7c
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation for...",None,"Yes, LangSmith supports offline evaluation thr...",1,1.924228,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11e0-4036-732d-b538-595ca4bde8b7
2,How can I trace with the @traceable decorator?,"To trace with the `@traceable` decorator, simp...",None,To trace with the @traceable decorator in Pyth...,1,2.869621,461abe28-c577-4c12-8577-85f68f4c44ab,019b11e0-47c3-7772-b47b-40a4106377f4
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",None,"Yes, LangSmith can be used to evaluate agents....",0,3.305131,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11e0-52fa-74a8-8e7f-ee8664999cdb
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,2.169151,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11e0-5fe6-7124-91ac-5cb7a996f4bd
5,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,2.414617,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11e0-686c-7521-8808-67202288fd90
6,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript LangSmith SDK. It a...",None,"Yes, thers is a JavaScript LangSmith SDK!",0,3.289736,6fd631fb-c1d0-4030-9511-5bcd93b980b8,019b11e0-71ed-7792-ba0c-6cf9bcc480dc
7,How do I create user feedback with the LangSmi...,You can create user feedback using the `create...,None,To create user feedback with the LangSmith SDK...,1,3.941710,0e14ba18-c937-4a43-b88b-678a1c5ea857,019b11e0-7ec6-754a-a39b-77cb22303c56
8,What testing capabilities does LangSmith have?,LangSmith offers prompt testing with built-in ...,None,LangSmith offers capabilities for creating dat...,1,2.547650,36415f3d-97ba-40a4-a75a-ecc563de722e,019b11e0-8e2e-723d-9355-a4da4c7ee35c
9,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,To set up tracing to LangSmith while using Lan...,0,3.375844,480d8cbc-c3d8-48d6-84ea-1366a59895be,019b11e0-9824-7601-94dc-5f3e0092f233


##### Concurrency

你还可以启动并发执行线程，使实验更快完成！

In [26]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="concurrency",
    max_concurrency=2,  # This defaults to None, so this is an improvement!
)

View the evaluation results for experiment: 'concurrency-5c66c91c' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=25750354-007d-4e6e-8c96-155dfaac3a49




12it [00:16,  1.40s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation for...",None,"Yes, LangSmith supports offline evaluation thr...",1,1.978168,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11e3-6b0e-7685-a58f-ace3d7168fe9
1,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,"To set up tracing to LangSmith with LangChain,...",1,2.970035,005e9c46-2975-43b6-b751-f64e69357e7b,019b11e3-6b0c-74f0-9620-b34a78f1729d
2,How can I trace with the @traceable decorator?,"To trace with the `@traceable` decorator, simp...",None,To trace with the @traceable decorator in Pyth...,1,2.876197,461abe28-c577-4c12-8577-85f68f4c44ab,019b11e3-72c9-739c-88fc-03f4b9d24d89
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",None,"Yes, LangSmith can be used to evaluate agents....",1,2.526872,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11e3-76a6-76c1-801e-1c33cc923bb2
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,1.960938,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11e3-7e06-71e3-b9ae-26191b1fe9d6
5,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,2.983644,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11e3-8085-7578-ad22-c75d1dcd1987
6,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript LangSmith SDK. It a...",None,"Yes, thers is a JavaScript LangSmith SDK!",0,2.539776,6fd631fb-c1d0-4030-9511-5bcd93b980b8,019b11e3-85af-7068-8948-e2bab0037639
7,How do I create user feedback with the LangSmi...,You can create user feedback using the `create...,None,To create user feedback with the LangSmith SDK...,1,2.989118,0e14ba18-c937-4a43-b88b-678a1c5ea857,019b11e3-8c32-715c-9fde-3a701c193ab9
8,What testing capabilities does LangSmith have?,LangSmith offers prompt testing with built-in ...,None,LangSmith offers capabilities for creating dat...,1,2.399178,36415f3d-97ba-40a4-a75a-ecc563de722e,019b11e3-8f9b-70fd-9db3-05d3faafa8dc
9,How do I pass metadata in with @traceable?,You can pass metadata statically by including ...,None,You can pass metadata with the @traceable deco...,1,2.312319,9f7c3004-23ab-4f45-9c97-4a28a8c6b53a,019b11e3-98fa-701c-9b33-f066327c71c6


##### Metadata 

你可以（也应该）为实验添加元数据，以便在用户界面中更容易找到它们。

In [27]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="metadata added",
    metadata={  # 我们可以传递实验的自定义元数据，例如模型名称。
        "model_name": MODEL_NAME
    }
)

View the evaluation results for experiment: 'metadata added-33c895ee' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/81e7503c-2a58-44fc-9d54-7bd9090c34e5/compare?selectedSessions=c5357d6f-ec57-4eab-ab9c-d4c13bfae0cb




12it [00:29,  2.50s/it]


,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,"To set up tracing to LangSmith with LangChain,...",1,2.827416,005e9c46-2975-43b6-b751-f64e69357e7b,019b11e4-69a5-7778-8d15-b2f89387de84
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation for...",None,"Yes, LangSmith supports offline evaluation thr...",1,1.874480,047b7fe4-eecb-41fb-88a0-57e8691d4e2d,019b11e4-74b3-73cb-b6b8-d43d52e232a2
2,How can I trace with the @traceable decorator?,"To trace with the `@traceable` decorator, simp...",None,To trace with the @traceable decorator in Pyth...,1,2.267974,461abe28-c577-4c12-8577-85f68f4c44ab,019b11e4-7c05-70dd-a62a-1e349bebfc65
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",None,"Yes, LangSmith can be used to evaluate agents....",1,2.715229,51983bd5-9b7b-4a4a-a121-14c76452c2a2,019b11e4-84e1-75ca-b208-32e2cb01c86d
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is not designed for fine-tuning ...",None,"Yes, LangSmith can be used for fine-tuning and...",1,2.101597,e220a991-071f-40a6-aa60-70c2cc58af2c,019b11e4-8f81-7197-b9d9-26ec9f88e60a
5,What is LangSmith used for in three sentences?,"LangSmith is used for developing, debugging, a...",None,LangSmith is a platform designed for the devel...,1,2.805357,e4d4a700-41e5-4dce-880c-c5fd17f26e75,019b11e4-97bb-7264-b3f6-da38b9725da2
6,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript LangSmith SDK. It a...",None,"Yes, thers is a JavaScript LangSmith SDK!",0,2.598861,6fd631fb-c1d0-4030-9511-5bcd93b980b8,019b11e4-a2c0-7644-afc5-c4ef67dd2c8c
7,How do I create user feedback with the LangSmi...,You can create user feedback using the `create...,None,To create user feedback with the LangSmith SDK...,1,2.661611,0e14ba18-c937-4a43-b88b-678a1c5ea857,019b11e4-ace7-7408-8e74-db2479de970c
8,What testing capabilities does LangSmith have?,LangSmith offers prompt testing with built-in ...,None,LangSmith offers capabilities for creating dat...,1,2.537678,36415f3d-97ba-40a4-a75a-ecc563de722e,019b11e4-b74d-7723-b17b-81be5cdef425
9,How do I set up tracing to LangSmith if I'm us...,"To set up tracing to LangSmith with LangChain,...",None,To set up tracing to LangSmith while using Lan...,1,2.560173,480d8cbc-c3d8-48d6-84ea-1366a59895be,019b11e4-c137-71b2-955d-5c691d3b26be
